In [15]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from transformers import XLMRobertaTokenizer

In [16]:
!pip install emoji
import emoji

In [17]:
# Bengali datasets are often saved as utf-8-sig
train_df = pd.read_csv("train.csv", encoding="utf-8-sig")
val_df   = pd.read_csv("validation.csv", encoding="utf-8-sig")
test_df  = pd.read_csv("test.csv", encoding="utf-8-sig")

In [18]:
# keeping only necessary columns
cols = ['Review', 'label']

train_df = train_df[cols].dropna()
val_df   = val_df[cols].dropna()
test_df  = test_df[cols].dropna()

Normalize **whitespace**

In [19]:
def clean_whitespace(text):
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text


Remove unnecessary **symbols**
1. keep English, Bengali, digits, basic punctuation
2. remove quotes

In [20]:
def remove_extra_symbols(text):
    text = str(text).strip()               # remove leading/trailing spaces
    text = re.sub(r'\s+', ' ', text)      # replace multiple spaces/newlines with a single space
    return text


Lowercase **English** text only

In [21]:
def conditional_lower(text):
    return ''.join([c.lower() if c.isascii() else c for c in text])

Convert **emojis** to text

In [22]:
def demojize_text(text):
    return emoji.demojize(text)

In [23]:
def preprocess_dataframe(df):
    df['Review'] = df['Review'].astype(str)
    df['Review'] = df['Review'].apply(clean_whitespace)
    df['Review'] = df['Review'].apply(remove_extra_symbols)
    df['Review'] = df['Review'].apply(conditional_lower)
    df['Review'] = df['Review'].apply(demojize_text)
    return df

In [25]:
train_df = preprocess_dataframe(train_df)
val_df   = preprocess_dataframe(val_df)
test_df  = preprocess_dataframe(test_df)

Verify label **consistency**

In [26]:
print("Train labels:\n", train_df['label'].value_counts())
print("Validation labels:\n", val_df['label'].value_counts())
print("Test labels:\n", test_df['label'].value_counts())

Train labels:
 label
2    99110
0     6772
1     4763
Name: count, dtype: int64
Validation labels:
 label
2    14159
0      967
1      680
Name: count, dtype: int64
Test labels:
 label
2    28318
0     1935
1     1361
Name: count, dtype: int64


**Tokenization** (XLM-RoBERTa)

In [27]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_texts(texts):
    return tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=256
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [28]:
train_encodings = tokenize_texts(train_df['Review'])
val_encodings   = tokenize_texts(val_df['Review'])
test_encodings  = tokenize_texts(test_df['Review'])

Final objects ready for **training**

In [29]:
train_labels = train_df['label'].tolist()
val_labels   = val_df['label'].tolist()
test_labels  = test_df['label'].tolist()